In [1]:
"""Why PAD / BOS / EOS exist, and what key_padding_mask actually does.
Three sentence pairs, real tensors, printed."""
import torch, torch.nn.functional as F
torch.manual_seed(0)
torch.set_printoptions(precision=3, sci_mode=False, linewidth=200)

PAD, BOS, EOS = 0, 1, 2
chars = "abcdefghijklmnopqrstuvwxyz ,.?!"
stoi = {c: i + 3 for i, c in enumerate(chars)}          # 0,1,2 are reserved
itos = {i: c for c, i in stoi.items()}
itos.update({PAD: '_', BOS: '>', EOS: '<'})
enc = lambda s: [stoi[c] for c in s.lower()]
dec = lambda ids: ''.join(itos[i] for i in ids)

pairs_en = ["go.",              "hello there.",       "i am an engineer."]
pairs_tr = ["git.",             "merhaba oradaki.",   "ben bir muhendisim."]

print("=" * 76)
print("1. THE PROBLEM: a batch is a RECTANGLE, sentences are not")
print("=" * 76)
raw_en = [enc(s) for s in pairs_en]
for s, r in zip(pairs_en, raw_en):
    print(f"   {len(r):2d} tokens  {s!r}")
try:
    torch.stack([torch.tensor(r) for r in raw_en])
except Exception as e:
    print(f"   torch.stack -> {type(e).__name__}: {e}")

print()
print("=" * 76)
print("2. THE FIX: BOS/EOS wrap the sequence, PAD fills to T_max")
print("=" * 76)


def pad_to(seqs, T):
    return torch.tensor([s + [PAD] * (T - len(s)) for s in seqs])


T_en = max(len(r) + 1 for r in raw_en)                 # +1 for EOS
src_en = pad_to([r + [EOS] for r in raw_en], T_en)

raw_tr = [enc(s) for s in pairs_tr]
T_tr = max(len(r) + 1 for r in raw_tr)                 # +1 for BOS / EOS
dec_in_tr = pad_to([[BOS] + r for r in raw_tr], T_tr)   # what the decoder READS
dec_out_tr = pad_to([r + [EOS] for r in raw_tr], T_tr)  # what it must PREDICT

print(f"   src_en      {tuple(src_en.shape)}")
for i in range(3):
    print(f"      {dec(src_en[i].tolist())!r}")
print(f"\n   dec_in_tr   {tuple(dec_in_tr.shape)}   (teacher forcing input)")
for i in range(3):
    print(f"      {dec(dec_in_tr[i].tolist())!r}")
print(f"\n   dec_out_tr  {tuple(dec_out_tr.shape)}   (the target, = input shifted LEFT by 1)")
for i in range(3):
    print(f"      {dec(dec_out_tr[i].tolist())!r}")
print("\n   position-by-position, row 2:")
print(f"      reads   {[itos[i] for i in dec_in_tr[2].tolist()]}")
print(f"      predict {[itos[i] for i in dec_out_tr[2].tolist()]}")

print()
print("=" * 76)
print("3. WHAT GOES WRONG WITHOUT key_padding_mask")
print("=" * 76)
pad_mask_en = (src_en == PAD)                           # [B, T_en]  True where PAD
print(f"   pad_mask_en {tuple(pad_mask_en.shape)}  (True = this KEY is padding)")
print(pad_mask_en.int())

B, T_q, T_k, Hs = 3, 4, T_en, 8
Q = torch.randn(B, T_q, Hs)
K = torch.randn(B, T_k, Hs)
scores = (Q @ K.transpose(-2, -1)) * Hs ** -0.5

unmasked = F.softmax(scores, dim=-1)
masked = F.softmax(scores.masked_fill(pad_mask_en[:, None, :], float('-inf')), dim=-1)

row = 0
print(f"\n   batch row 0 is {dec(src_en[0].tolist())!r}: "
      f"{int((~pad_mask_en[0]).sum())} real tokens, {int(pad_mask_en[0].sum())} PAD")
print(f"   one query's attention over those {T_k} keys:")
print(f"      WITHOUT mask {unmasked[row, 0]}")
print(f"      WITH    mask {masked[row, 0]}")
w_leak = unmasked[row, :, pad_mask_en[row]].sum(-1)
print(f"\n   probability mass LEAKED onto PAD (per query, row 0):")
print(f"      without mask {w_leak}")
print(f"      with    mask {masked[row][:, pad_mask_en[row]].sum(-1)}")
print(f"   -> {w_leak.mean()*100:.1f}% of every query's attention went to filler.")
print(f"      Those PAD columns are V-vectors of the embedding of token id 0,")
print(f"      which is a LEARNED vector -- so the model trains on garbage that")
print(f"      changes with the batch's longest sentence. Silent, never crashes.")

print()
print("=" * 76)
print("4. THE SAME IDEA ON THE LOSS SIDE: ignore_index")
print("=" * 76)
V = len(itos)
logits = torch.randn(B, T_tr, V)
naive = F.cross_entropy(logits.reshape(-1, V), dec_out_tr.reshape(-1))
right = F.cross_entropy(logits.reshape(-1, V), dec_out_tr.reshape(-1), ignore_index=PAD)
n_pad = int((dec_out_tr == PAD).sum())
print(f"   {n_pad} of {dec_out_tr.numel()} target positions are PAD "
      f"({n_pad/dec_out_tr.numel()*100:.0f}%)")
print(f"   loss without ignore_index : {naive:.4f}   <- trains the model to emit PAD")
print(f"   loss with    ignore_index : {right:.4f}")

1. THE PROBLEM: a batch is a RECTANGLE, sentences are not
    3 tokens  'go.'
   12 tokens  'hello there.'
   17 tokens  'i am an engineer.'
   torch.stack -> RuntimeError: stack expects each tensor to be equal size, but got [3] at entry 0 and [12] at entry 1

2. THE FIX: BOS/EOS wrap the sequence, PAD fills to T_max
   src_en      (3, 18)
      'go.<______________'
      'hello there.<_____'
      'i am an engineer.<'

   dec_in_tr   (3, 20)   (teacher forcing input)
      '>git._______________'
      '>merhaba oradaki.___'
      '>ben bir muhendisim.'

   dec_out_tr  (3, 20)   (the target, = input shifted LEFT by 1)
      'git.<_______________'
      'merhaba oradaki.<___'
      'ben bir muhendisim.<'

   position-by-position, row 2:
      reads   ['>', 'b', 'e', 'n', ' ', 'b', 'i', 'r', ' ', 'm', 'u', 'h', 'e', 'n', 'd', 'i', 's', 'i', 'm', '.']
      predict ['b', 'e', 'n', ' ', 'b', 'i', 'r', ' ', 'm', 'u', 'h', 'e', 'n', 'd', 'i', 's', 'i', 'm', '.', '<']

3. WHAT GOES WRONG WITH